# SmolVLA LoRA Training

Fine-tune SmolVLA (450M) with LoRA on LIBERO-Object. Optimized for Kaggle T4 16GB.

In [ ]:
# Cell 1: Install and setup
!pip install -q lerobot[smolvla,peft,libero]
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Train SmolVLA with LoRA
import subprocess, time, os

# Pass required env vars to subprocess (LIBERO non-interactive + headless rendering)
train_env = {
    **os.environ,
    "MUJOCO_GL": "egl",
    "LIBERO_BENCHMARK_PATH": "/kaggle/working/libero_benchmarks",
    "TOKENIZERS_PARALLELISM": "false",
}

start = time.time()
proc = subprocess.Popen([
    "lerobot-train",
    "--policy.path=lerobot/smolvla_base",
    "--dataset.repo_id=HuggingFaceVLA/libero",
    "--policy.repo_id=smolvla-libero-object-lora",
    "--batch_size=8",
    "--steps=20000",
    "--save_checkpoint=true",
    "--log_freq=100",
    "--peft.method_type=LORA",
    "--peft.r=32",
    "--optimizer.lr=1e-3",
    "--wandb.enable=false",
    # Map dataset camera keys → policy camera names (smolvla_base expects camera1/2/3)
    '--rename_map={"observation.images.image": "observation.images.camera1", "observation.images.image2": "observation.images.camera2"}',
    # Allow missing camera3 to be filled with a zero tensor (SmolVLA empty_cameras mechanism)
    "--policy.empty_cameras=1",
], env=train_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Stream output live
for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait(timeout=36000)
elapsed = time.time() - start
print(f"\nTraining exit code: {proc.returncode}")
print(f"Time elapsed: {elapsed/3600:.1f} hours")

In [ ]:
# Cell 3: Save and push checkpoint
import glob
from huggingface_hub import HfApi

# Find latest checkpoint
checkpoints = sorted(glob.glob("outputs/*/checkpoints/*"))
print(f"Checkpoints found: {checkpoints}")

if checkpoints:
    latest = checkpoints[-1]
    print(f"Latest checkpoint: {latest}")

    # Optional: push to Hub (set your HF token first)
    # api = HfApi()
    # api.upload_folder(
    #     folder_path=latest,
    #     repo_id="YOUR_USERNAME/smolvla-libero-object-lora",
    #     repo_type="model"
    # )
    # print("Pushed to Hub!")

In [ ]:
import os
# Cell 4: Plot training loss from logs
import json
import matplotlib.pyplot as plt

# LeRobot logs training metrics to stdout/files — parse them
# Location depends on LeRobot version; adapt path as needed
log_files = glob.glob("outputs/*/logs/*.json") + glob.glob("outputs/*/train_log.jsonl")
print(f"Log files: {log_files}")

if log_files:
    losses = []
    steps = []
    with open(sorted(log_files, key=os.path.getmtime)[-1]) as f:
        for line in f:
            try:
                entry = json.loads(line)
                if "loss" in entry:
                    losses.append(entry["loss"])
                    steps.append(entry.get("step", len(steps)))
            except json.JSONDecodeError:
                continue

    if losses:
        plt.figure(figsize=(10, 4))
        plt.plot(steps, losses)
        plt.xlabel("Step")
        plt.ylabel("Loss")
        plt.title("SmolVLA LoRA Training Loss")
        plt.grid(True, alpha=0.3)
        plt.savefig("training_loss.png", dpi=100)
        plt.show()
        print(f"Final loss: {losses[-1]:.4f}")